In [ ]:
from cemp_software_settings import load_and_apply_settings
result = load_and_apply_settings()
print(result)

In [ ]:
from rdkit.Chem import AllChem
import os
import re
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AddHs
from rdkit.Chem.rdmolfiles import MolToSmiles
from openbabel import openbabel, pybel
from rdkit.Chem import RemoveHs
import shutil

In [ ]:

df = pd.read_excel('Dimer.xlsx')
dimer_excel_path = 'Dimer.xlsx'

In [ ]:

nproc = 10
mem = 2000 

In [ ]:
df

In [ ]:

def is_monoatomic_ion(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol.GetNumAtoms() == 1

In [ ]:


def calculate_charge_and_multiplicity(smiles):
    
    mol = Chem.MolFromSmiles(smiles)
    
    
    mol = Chem.AddHs(mol)
    
    
    charge = sum(atom.GetFormalCharge() for atom in mol.GetAtoms())
    
    
    multiplicity = sum([atom.GetNumRadicalElectrons() for atom in mol.GetAtoms()]) + 1
    
    return charge, multiplicity

In [ ]:
def create_ORCA_inputfile(name, smiles):
    
    mol = Chem.MolFromSmiles(smiles)

    
    smiles_noH = MolToSmiles(mol, allHsExplicit=False)

    
    obConversion = openbabel.OBConversion()
    obConversion.SetInAndOutFormats("smi", "gjf")
    mol = pybel.readstring("smi", smiles_noH)
    mol.addh()
    mol.make3D(forcefield='mmff94', steps=50)

    
    filename = name.replace(' ', '_') + '.inp'

    
    obConversion.WriteFile(mol.OBMol, filename)

    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')

    with open(filename, 'r') as file:
        lines = file.readlines()

    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index
            break

    
    if start_index is not None:
        with open(filename, 'w') as file:
            file.writelines(lines[start_index:])

    
    
    with open(filename, 'r') as f:
        coordinates = f.read()    
    
    
    
    charge, multiplicity = calculate_charge_and_multiplicity(smiles)
    

    
    with open(filename, 'w') as f:
        
        if is_monoatomic_ion(smiles):
            f.write(f"! wB97M-V def2-TZVP def2/J RIJCOSX strongSCF noautostart miniprint\n")
        else:
            f.write(f"! B3LYP D3 def2-TZVP def2/J RIJCOSX opt freq tightSCF noautostart miniprint\n")
        f.write(f"%maxcore     {mem}\n")
        f.write(f"%pal nprocs  {nproc} end\n")
        f.write(f"* xyz   {charge} {multiplicity}\n")
        f.write(coordinates)
        f.write(f"*")

    
    
    if is_monoatomic_ion(smiles):
        xyz_fake_filename = name.replace(' ', '_') + '.xyz'
        with open(xyz_fake_filename, 'w') as f:
            f.write(coordinates)

In [ ]:

def create_or_load_molecule_xlsx(directory):
    
    xlsx_path = os.path.join(directory, 'molecule.xlsx')
    
    
    if not os.path.exists(xlsx_path):
        
        df = pd.DataFrame(columns=['FileName', 'SMILES'])
        
        df.to_excel(xlsx_path, index=False)
    else:
        
        df = pd.read_excel(xlsx_path)
    
    return df

In [ ]:

def get_filename_without_extension(xlsx_path):
    
    
    base_name = os.path.basename(xlsx_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:

def normalization(mol):
    smi=Chem.MolToSmiles(mol)
    n_mol=Chem.MolFromSmiles(smi)
    return n_mol

In [ ]:

def normalization_SMILES(df, SMILESname="SMILES"):
    
    if f"{SMILESname}" in df.columns:
        
        invalid_rows = []
        
        for index, row in df.iterrows():
            try:
                
                mol = Chem.MolFromSmiles(row[SMILESname])
                if mol:  
                    
                    n_mol = normalization(mol)
                    if n_mol:  
                        
                        n_smi = Chem.MolToSmiles(n_mol)
                        
                        df.at[index, SMILESname] = n_smi
                    else:
                        
                        invalid_rows.append(index)
                else:
                    
                    invalid_rows.append(index)
            except Exception as e:
                
                print(f"An error occurred for index {index}: {e}")
                invalid_rows.append(index)
        
        
        df.drop(invalid_rows, inplace=True)
        return df
        
    else:
        print(f"The {SMILESname} column does not exist in the provided Excel file.")

In [ ]:

def clean_name_column(df, name_col):
    

    
    illegal_chars = [",", ".", "/", "\\", " "]

    
    
    pattern = '|'.join([f"\\{char}" for char in illegal_chars])

    
    
    df[f'{name_col}'] = df[f'{name_col}'].str.replace(pattern, "_", regex=True)

    
    return df

In [ ]:

def read_excel_to_dict(excel_path, key_col, value_col):
    
    try:
        
        df = pd.read_excel(excel_path)
        
        
        if key_col not in df.columns or value_col not in df.columns:
            raise ValueError("Public message removed for release.")
        
        
        result_dict = pd.Series(df[value_col].values,index=df[key_col]).to_dict()
        
        return result_dict
    except FileNotFoundError:
        print("Public message removed for release.")
    except ValueError as e:
        print(e)
    except Exception as e:
        print("Public message removed for release.")

In [ ]:

def drop_duplicates(df,SMILESname="SMILES"):
    
    
    df.drop_duplicates(subset=SMILESname, keep='first', inplace=True)
    
    print("Public message removed for release.")

In [ ]:

def get_unique_filenames(directory):
    
    filenames = set(os.path.splitext(file)[0] for file in os.listdir(directory))
    return filenames

In [ ]:
import os  


def check_and_create_orca_database(base_dir: str = result['orca_database_path']):  
    
    home_directory = os.path.abspath(os.path.expanduser(base_dir))  
    orca_database_path = os.path.join(home_directory, 'ORCA_database')  
    optfreq_orca_database_path = os.path.join(orca_database_path, 'opt+freq')  
    RESPpolymer_database_path = os.path.join(orca_database_path, 'RESPpolymer')  

    if not os.path.exists(orca_database_path):  
        os.makedirs(orca_database_path)  
        os.makedirs(optfreq_orca_database_path)  
        os.makedirs(RESPpolymer_database_path)  
        print("Public message removed for release.")  

    elif not os.path.exists(optfreq_orca_database_path) and os.path.exists(RESPpolymer_database_path):  
        os.makedirs(optfreq_orca_database_path)  
        print("Public message removed for release.")  

    elif not os.path.exists(RESPpolymer_database_path) and os.path.exists(optfreq_orca_database_path):  
        os.makedirs(RESPpolymer_database_path)  
        print("Public message removed for release.")  

    elif not os.path.exists(RESPpolymer_database_path) and not os.path.exists(optfreq_orca_database_path):  
        os.makedirs(optfreq_orca_database_path)  
        os.makedirs(RESPpolymer_database_path)  
        print("Public message removed for release.")  

    else:  
        print("Public message removed for release.")  

    return orca_database_path, optfreq_orca_database_path, RESPpolymer_database_path  

In [ ]:


def compare_smiles_dicts(system_dict, database_dict):
    
    not_found_molecule_dict = {}
    found_molecule_dict = {}

    
    for smiles, name in system_dict.items():
        if smiles not in database_dict.keys():
            not_found_molecule_dict[smiles] = name

    
    for smiles in database_dict:
        if smiles in system_dict:
            name = system_dict[smiles]  
            found_molecule_dict[smiles] = name

    return not_found_molecule_dict, found_molecule_dict

In [ ]:

def read_system_excel_to_dict(excel_path, SMILESName="SMILES", Name="Name"):
    
    return read_excel_to_dict(excel_path, SMILESName, Name)


def read_database_excel_to_dict(database_path):
    
    return read_excel_to_dict(database_path, 'SMILES', 'FileName')

In [ ]:
def copy_files_based_on_smiles(database_directory, database_excel_path, found_molecule_dict, destination_directory):
    
    
    if not os.path.exists(destination_directory):
        raise Exception(f"{destination_directory} does not exist")

    
    df = pd.read_excel(database_excel_path)

    
    for smiles, name in found_molecule_dict.items():
        
        matched_files = df[df['SMILES'] == smiles]['FileName'].tolist()
        
        
        for matched_file in matched_files:
            
            for ext in ['.inp', '.out', '.opt', '.xyz', '.txt', '.gbw', '.engrad', '.densitiesinfo', '.densities', '.hess', '.bibtex']:
                file_to_copy = matched_file + ext
                source_path = os.path.join(database_directory, file_to_copy)
                
                if os.path.isfile(source_path):
                    
                    new_name = name + ext
                    destination_path = os.path.join(destination_directory, new_name)
                    
                    shutil.copy2(source_path, destination_path)
                    print("Public message removed for release.")
                else:
                    print("Public message removed for release.")

In [ ]:

df = clean_name_column(df, name_col="Dimer Name")
df = clean_name_column(df, name_col="Component Name A")
df = clean_name_column(df, name_col="Component Name B")


df = normalization_SMILES(df, SMILESname="Component SMILES A")
df = normalization_SMILES(df, SMILESname="Component SMILES B")
df = normalization_SMILES(df, SMILESname="Dimer SMILES")


drop_duplicates(df, SMILESname="Dimer SMILES")


df.to_excel('Dimer.xlsx', index=False)

In [ ]:
from qc_database_utils import (
    add_and_normalize_smiles,
    compare_smiles_dicts,
    copy_files_based_on_smiles,
    ensure_database_excel,
    get_orca_database_paths,
    read_database_excel_to_dict,
)

if 'result' not in globals():
    from cemp_software_settings import load_and_apply_settings
    result = load_and_apply_settings()


In [ ]:



orca_database_path, optfreq_orca_database_path, RESPpolymer_database_path = get_orca_database_paths(result)


molecule_database_path = optfreq_orca_database_path
polymer_database_path = RESPpolymer_database_path


molecule_database_excel_path = ensure_database_excel(molecule_database_path)
polymer_database_excel_path = ensure_database_excel(polymer_database_path)


molecule_database = read_database_excel_to_dict(molecule_database_excel_path)  
polymer_database = read_database_excel_to_dict(polymer_database_excel_path)  


In [ ]:

componentA_system_dict = read_system_excel_to_dict(dimer_excel_path, SMILESName="Component SMILES A", Name="Component Name A")
componentB_system_dict = read_system_excel_to_dict(dimer_excel_path, SMILESName="Component SMILES B", Name="Component Name B")


componentA_not_found_dict, componentA_found_dict = compare_smiles_dicts(componentA_system_dict, molecule_database)
componentB_not_found_dict, componentB_found_dict = compare_smiles_dicts(componentB_system_dict, molecule_database)

In [ ]:

os.makedirs('component_gas/opt+freq/success', exist_ok=True)
os.makedirs('component_gas/opt+freq/failure', exist_ok=True)

os.makedirs("component_gas/opt+freq/imaginary_frequency", exist_ok=True)

base_dir = 'component_gas/opt+freq'
success_dir = os.path.join(base_dir, 'success') 
failure_dir = os.path.join(base_dir, 'failure') 
imaginary_frequency_dir = os.path.join(base_dir, "imaginary_frequency") 

In [ ]:

copy_files_based_on_smiles(molecule_database_path, molecule_database_excel_path, componentA_found_dict, success_dir)
copy_files_based_on_smiles(molecule_database_path, molecule_database_excel_path, componentB_found_dict, success_dir)

In [ ]:


def create_input_files_for_missing_molecules(not_found_molecule_dict):
    
    
    for smiles, name in not_found_molecule_dict.items():
        
        
        create_ORCA_inputfile(name, smiles)

In [ ]:
create_input_files_for_missing_molecules(componentA_not_found_dict)
create_input_files_for_missing_molecules(componentB_not_found_dict)